In [1]:
from statsmodels.tsa.api import VAR
from statsmodels.api import OLS, add_constant

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import jpype
import os
from jpype import JClass
jar_path = os.path.abspath("infodynamics.jar")
jpype.startJVM(classpath=[jar_path])
TECalcClass = jpype.JPackage("infodynamics.measures.continuous.kraskov").TransferEntropyCalculatorKraskov
teCalc = TECalcClass()
from jpype import JPackage
CTEClass = JPackage("infodynamics.measures.continuous.kraskov").ConditionalTransferEntropyCalculatorKraskov
cteCalc = CTEClass()
MIClass = jpype.JPackage("infodynamics.measures.continuous.kraskov").MutualInfoCalculatorMultiVariateKraskov1
miCalc = MIClass()

## Igualdad entre Granger Causality y Transfer Entropy en modelos lineales bajo gaussianidad

Para ilustrar de forma analítica la equivalencia teórica demostrada, procedemos directamente con el proceso bivariante VAR($\tau$) con acoplamiento $Y \to X$:
$$
\begin{cases}
X_t = a X_{t-1} + b Y_{t-\tau} + \varepsilon_t^x \\
Y_t = c Y_{t-1} + \varepsilon_t^y,
\end{cases}
$$
donde los términos de error (ruido blanco) son independientes y siguen una distribución normal:
$$ \varepsilon_t^x \sim \mathcal{N}(0, \sigma_x^2) \quad \text{y} \quad \varepsilon_t^y \sim \mathcal{N}(0, \sigma_y^2), $$
con $|a| < 1$ y $|c| < 1$ para garantizar la estacionariedad.

Observar que el anterior modelo cumple todas las hipótesis bajo las cuales la Granger Causality (GC) y la Transfer Entropy (TE) son equivalentes: linealidad del sistema, normalidad multivariante, estacionariedad y ruido blanco incorrelado.

**Modelo completo.** Si conocemos $X_{t-1}$ e $Y_{t-\tau}$, el error de predicción de $X_t$ es el ruido blanco puro:
$$\mathrm{Var}(X_t \mid X_{t-1}, Y_{t-\tau}) = \sigma_x^2.$$

**Modelo reducido.** Si solo conocemos $X_{t-1}$, el término $b\,Y_{t-\tau}$ pasa a formar parte del error:
$$\mathrm{Var}(X_t \mid X_{t-1}) = b^2 \, \mathrm{Var}(Y_{t-\tau} \mid X_{t-1}) + \sigma_x^2,
\qquad
\mathrm{Var}(Y_{t-\tau} \mid X_{t-1}) = V_y - \frac{\mathrm{Cov}(Y_{t-\tau}, X_{t-1})^2}{V_x}.$$

Basta, por tanto, con calcular tres cantidades estacionarias:

- La varianza de la fuente, que es un AR(1) independiente:
$$V_y = \frac{\sigma_y^2}{1 - c^2}.$$
- La covarianza entre la fuente en su retardo y el pasado inmediato del destino. Desarrollando $X_{t-1} = \sum_{j \geq 0} a^j \left( b\, Y_{t-1-j-\tau} + \varepsilon^x_{t-1-j} \right)$ y usando $\mathrm{Cov}(Y_{t-\tau}, Y_{t-\tau-1-j}) = c^{\,j+1} V_y$:
$$\mathrm{Cov}(Y_{t-\tau}, X_{t-1}) = \sum_{j \geq 0} a^j\, b\, c^{\,j+1} V_y = \frac{b\,c}{1 - a c}\, V_y.$$
- La varianza del destino:
$$V_x = \frac{1}{1 - a^2} \left[ \left( \frac{2 a b^2 c}{1 - a c} + b^2 \right) V_y + \sigma_x^2 \right].$$

Ninguna de las tres depende de $\tau$: el retardo únicamente desplaza la fuente en el tiempo (basta redefinir $Y'_t = Y_{t-\tau+1}$ para recuperar el VAR(1)). En consecuencia, cuando la historia del destino se limita a $X_{t-1}$ y la fuente se evalúa exactamente en su retardo estructural, el valor teórico de la TE y de la GC es el mismo para cualquier $\tau \geq 1$.

Sustituyendo en la expresión que define la Transfer Entropy como cociente de varianzas:
$$TE_{Y \to X} = \frac{1}{2} \ln \left( \frac{b^2 \, \mathrm{Var}(Y_{t-\tau} \mid X_{t-1}) + \sigma_x^2}{\sigma_x^2} \right).$$
A partir de esta igualdad, y utilizando la identidad $F_{Y \to X} = 2\, TE_{Y \to X}$, se obtiene del mismo modo el valor continuo de la GC.

De acuerdo a la teoría anterior, a continuación se define la función `GC_TE_autorreg_tau_gauss` que calcula los valores de Granger Causality y Transfer Entropy teóricos asociados a un modelo Autorregresivo lineal Gaussiano (VAR($\tau$) bivariante).

La función `GC_TE_autorreg_tau_gauss` recive los parámetros del sistema Autorregresivo y devuelve las cantidades $F_{Y \to X}$ y $TE_{Y \to X} $. Existe la opción de imprimir por salida los elementos fundamentales de la construcción matemática que componen la matriz de covarianza $
\Sigma = 
\begin{pmatrix} 
\text{Var}(X) & \text{Cov}(X,Y) \\ 
\text{Cov}(X,Y) & \text{Var}(Y) 
\end{pmatrix}
= \begin{pmatrix} V_x & C_{xy} \\ C_{xy} & V_y \end{pmatrix}
$.

In [81]:
def GC_TE_autorreg_tau_gauss(a, b, c, sig_x2, sig_y2, tau, datos=True):
    """
    Calcula la Granger Causality F y la Transfer Entropy TE
    usando las fórmulas analíticas derivadas del modelo VAR(tau).
                  Y --> X
    """
    Vy = sig_y2 / (1 - c**2)   
    Cxy = (b * c**(tau-1) / (1 - a * c)) * Vy    
    termino_intermedio = ((2 * a * b**2 * c**tau) / (1 - a * c)) + b**2
    Vx = (1 / (1 - a**2)) * (termino_intermedio * Vy + sig_x2)         
    
    Var_Y_cond_X = Vy - (Cxy**2 / Vx)  
    
    F = np.log((b**2 * Var_Y_cond_X + sig_x2) / sig_x2) 
    TE = 0.5 * F 

    if datos:
        return {"Vx": Vx, "Vy": Vy, "Cxy": Cxy, "Var(Y|X)": Var_Y_cond_X, "Granger_F": F, "Transfer_Entropy": TE}
    else:
        return F, TE

In [23]:
def GC_TE_autorreg_tau_gauss(a, b, c, sig_x2, sig_y2, tau=None, datos=True):
    """
    GC (F) y TE teóricas del VAR(tau) gaussiano con acoplamiento Y --> X:
    X_t = a X_{t-1} + b Y_{t-tau} + eps_x
    Y_t = c Y_{t-1} + eps_y
    Supone historia del destino de 1 paso y la fuente en su retardo exacto.
    El resultado NO depende de tau (se mantiene el argumento por compatibilidad).
    """
    Vy = sig_y2 / (1 - c**2)
    Cov_X1_Ytau = (b * c / (1 - a * c)) * Vy 
    termino_intermedio = (2 * a * b**2 * c) / (1 - a * c) + b**2
    Vx = (termino_intermedio * Vy + sig_x2) / (1 - a**2)
    
    Var_Y_cond_X = Vy - Cov_X1_Ytau**2 / Vx 
    
    F = np.log((b**2 * Var_Y_cond_X + sig_x2) / sig_x2)
    TE = 0.5 * F
    if datos:
        return {"Vx": Vx, "Vy": Vy, "Cov(X_{t-1},Y_{t-tau})": Cov_X1_Ytau,
        "Var(Y|X)": Var_Y_cond_X, "Granger_F": F, "Transfer_Entropy": TE}
    return F, TE

A continuación se muestra un ejemplo de uso de la anterior función:

In [14]:
# Definición de parámetros teóricos
a, b, c = 0.5, 0.3, 0.7
sig_x2, sig_y2 = 1.0, 1.0
tau = 3

In [15]:
resultados_teóricos = GC_TE_autorreg_tau_gauss(a=a, b=b, c=c, sig_x2=sig_x2, sig_y2=sig_y2, tau=tau)

print(f"--- Resultados del Análisis Teórico ---")
for k, v in resultados_teóricos.items():
    print(f"{k:18}: {v:.6f}")

--- Resultados del Análisis Teórico ---
Vx                : 1.822021
Vy                : 1.960784
Cov(X_{t-1},Y_{t-tau}): 0.633484
Var(Y|X)          : 1.740533
Granger_F         : 0.145526
Transfer_Entropy  : 0.072763


En este notebook se evaluan los estimadores de Granger Causality y Transfer Entropy utilizando como valores "ground truth" los obtenidos teóricamente mediante la función `GC_TE_autorreg_tau_gauss`. De este modo se evalúa si en la práctica ambos estimadores son equivalentes bajo linealidad y perturbaciones gaussianas, y si bajo esos supuestos los valores estimados concuerdan con los valores teóricos.

### Granger Causality

Comenzamos evaluando el estimador de GC que se usa en este trabajo; el recurso `VAR`de la librería `statsmodels`.

Se usarán simulaciones de un modelo VAR($\tau$) como el siguiente:

In [16]:
# Definición de parámetros teóricos
a, b, c = 0.5, 0.3, 0.7
sig_x2, sig_y2 = 1.0, 1.0
tau = 6
T = 10000

ex = np.random.normal(0, np.sqrt(sig_x2), T)
ey = np.random.normal(0, np.sqrt(sig_y2), T)

Z = np.zeros((T, 2)) # Z[:, 0] es X, Z[:, 1] es Y

for t in range(tau, T):
    # Ecuación para Y: Solo depende de su propio pasado inmediato
    Z[t, 1] = c * Z[t-1, 1] + ey[t]
    
    # Ecuación para X: Depende de su pasado inmediato y de Y con delay tau
    Z[t, 0] = a * Z[t-1, 0] + b * Z[t-tau, 1] + ex[t]

# Ajuste del modelo VAR
data = pd.DataFrame(Z, columns=['X', 'Y'])
modelo_var = VAR(data)

La herramienta permite identificar explícitamente los coeficientes del modelo y la matriz de covarianzas:

In [17]:
# Ajuste del modelo VAR(tau)
modelo = modelo_var.fit(tau)

# Coeficientes estimados (aproximan a, b, 0, c)
print(f"--- Matriz de parámetros ---")
print(modelo.params)

# Matriz de covarianza de residuos (debe aproximarse a Sigma_e)
print(f"--- Matriz de Covarianza ---")
print(modelo.sigma_u)

--- Matriz de parámetros ---
              X         Y
const -0.003640 -0.014088
L1.X   0.505144 -0.004905
L1.Y  -0.004690  0.715242
L2.X  -0.009622 -0.005377
L2.Y   0.003318 -0.004292
L3.X   0.016822  0.008258
L3.Y  -0.016542 -0.013586
L4.X  -0.005082 -0.000196
L4.Y   0.012333  0.019440
L5.X   0.015890 -0.010486
L5.Y  -0.021075 -0.000641
L6.X  -0.009211  0.012648
L6.Y   0.299561  0.001867
--- Matriz de Covarianza ---
          X         Y
X  0.990229  0.013525
Y  0.013525  0.982861


Idealmente, la herramienta está diseñada para realizar el test de significación para dedicir si existe causalidad o no mediante el criterio de p-valor.

In [18]:
causality_test = modelo.test_causality(caused='X', causing=['Y'], kind='f')
print(causality_test.summary())
print(causality_test.pvalue)          

Granger causality F-test. H_0: Y does not Granger-cause X. Conclusion: reject H_0 at 5% significance level.
Test statistic Critical value p-value     df    
------------------------------------------------
         245.2          2.099   0.000 (6, 19962)
------------------------------------------------
1.3311147772847332e-303


Sin embargo, como ya se comentó, en este trabajo se está igualmente interesado por el valor de la intensidad de enlace que se puede calcular mediante la versión contínua de Geweke: 
$$
F_{Y \to X} = \log \left( 
\frac{\text{Var}(X_t \mid X_{t-1})}
{\text{Var}(X_t \mid X_{t-1}, Y_{t-1})}
\right).
$$
La siguiente función `gc_statsmodels` utiliza instrumentos internos de `statsmodels` para calcular dicho valor contínuo.

In [19]:
def gc_statsmodels(V1, V2, lag=1):
    """
    Calcula F_{Y -> X} = log( Var(X_t | X_{t-1}) / Var(X_t | X_{t-1}, Y_{t-1}) )

    Parámetros
    ----------
    V2 : np.ndarray
        Serie X (variable efecto)
    V1 : np.ndarray
        Serie Y (variable causa)
    lag : int
        Número de retardos (por defecto 1)

    Returns
    -------
    GC : float
        Granger causality (log-ratio de varianzas)
    """

    data = pd.DataFrame({'X': V2, 'Y': V1})
    modelo_var = VAR(data)
    modelo = modelo_var.fit(lag)

    # Varianza condicional completa Var(X_t | X_{t-1}, Y_{t-1})
    sigma_full = modelo.sigma_u.loc['X', 'X'] 

    # Modelo reducido (OLS)
    X_t = V2[lag:]
    X_lags = np.column_stack([V2[lag - i - 1: -i - 1] for i in range(lag)]) 
    X_lags = add_constant(X_lags)   
    modelo_restr = OLS(X_t, X_lags).fit() 
    sigma_restr = modelo_restr.mse_resid

    GC = np.log(sigma_restr / sigma_full)

    return GC

Con eso, vamos a comprobar el ajuste de la estimación a su valor teórico para el anterior modelo VAR(3). Para obtener resultados robustos al evaluar la precisión y consistencia del estimador de Granger Causality en condiciones estocásticas controladas, se implementa un experimento estructurado en $20$ realizaciones independientes sobre series temporales de longitud $T = 10000$ del proceso autorregresivo bivariado. Sobre estas trayectorias, se calcula numéricamente la Granger Causality teórica bajo el supuesto gaussiano y se contrasta frente a las estimaciones empíricas obtenidas tanto en la dirección causal directa ($Y \to X$) como en la inversa ($X \to Y$). Este diseño permite cuantificar el error relativo medio de la estimación frente al valor analítico de referencia.

In [24]:
a, b, c = 0.5, 0.3, 0.7
sig_x2, sig_y2 = 1.0, 1.0
tau = 6
T = 10000
num_realizaciones = 20

lista_F_teorica = []
lista_F_estimada = []
lista_F_inversa = []
lista_errores_relativos = []

for i in range(num_realizaciones):
    ex = np.random.normal(0, np.sqrt(sig_x2), T)
    ey = np.random.normal(0, np.sqrt(sig_y2), T)
    Z = np.zeros((T, 2))
    for t in range(tau, T):
        Z[t, 1] = c * Z[t-1, 1] + ey[t]
        Z[t, 0] = a * Z[t-1, 0] + b * Z[t-tau, 1] + ex[t]
    data = pd.DataFrame(Z, columns=['X', 'Y'])
    
    F_estimada = gc_statsmodels(data['Y'].values, data['X'].values, lag=tau)
    F_estimada_inversa = gc_statsmodels(data['X'].values, data['Y'].values, lag=tau)
    F_teorica, TE_teorica = GC_TE_autorreg_tau_gauss(a=a, b=b, c=c, sig_x2=sig_x2, sig_y2=sig_y2, tau=tau, datos=False)

    error_relativo_iter = abs(F_estimada - F_teorica) / F_teorica

    lista_F_teorica.append(F_teorica)
    lista_F_estimada.append(F_estimada)
    lista_F_inversa.append(F_estimada_inversa)
    lista_errores_relativos.append(error_relativo_iter)

media_F_teorica = np.mean(lista_F_teorica)
media_F_estimada = np.mean(lista_F_estimada)
media_F_inversa = np.mean(lista_F_inversa)
error_relativo_medio = np.mean(lista_errores_relativos)

print(f"--- Resultados medios tras {num_realizaciones} realizaciones ---")
print(f"F teórica media: {media_F_teorica:.4f}")
print(f"F estimada inversa media (VAR Statsmodels): {media_F_inversa:.4f}")
print(f"F estimada media (VAR Statsmodels): {media_F_estimada:.4f}")
print(f"Error relativo medio (promedio de los errores): {error_relativo_medio:.4%}")

--- Resultados medios tras 20 realizaciones ---
F teórica media: 0.1455
F estimada inversa media (VAR Statsmodels): 0.0000
F estimada media (VAR Statsmodels): 0.1469
Error relativo medio (promedio de los errores): 2.3206%


### Transfer Entropy

Pasamos ahora a la evaluación del estimador de Transfer Entropy; JIDT de Google Code. La siguiente funión; `TE_JIDT`, devuelve el valor contínuo de la intensidad de TE.

In [25]:
def TE_JIDT(source, target, contexto_k=1, contexto_l=1, contexto_c=1, n_neighbors=4, tau_source=1):
    """
    Calcula la Transfer Entropy entre dos series temporales mediante la herramienta JIDT.
    
    Parámetros:
    -----------
    source : array-like
        Serie temporal de origen.
    target : array-like
        Serie temporal de destino.
    contexto_k : int
        Historial para la variable de destino (k_HISTORY).
    contexto_l : int
        Historial para la variable de origen (l_HISTORY).
    contexto_c : int
        Historial de contexto/condicional.
    n_neighbors : int
        Número de vecinos cercanos (k) para el estimador KSG.
    tau : int
        Retardo temporal (DELAY).
        
    Retorna:
    --------
    float
        Valor de la Transfer Entropy calculada.
    """
    hist = max(contexto_k, contexto_l, contexto_c)
    
    source_trimmed = source[hist:]
    target_trimmed = target[hist:]
    
    teCalc = TECalcClass()
    teCalc.setProperty("k", str(n_neighbors))
    teCalc.setProperty("k_HISTORY", str(contexto_k))
    teCalc.setProperty("l_HISTORY", str(contexto_l))    
    teCalc.setProperty("DELAY", str(tau))
    teCalc.initialise()
    
    teCalc.setObservations(source_trimmed.astype(np.float64), target_trimmed.astype(np.float64))
    te_ksg_val = teCalc.computeAverageLocalOfObservations()
    
    return te_ksg_val

Se utilizará el sistema VAR($\tau$) comentado anteriormente. En este caso $\tau=6$:

$$
\begin{cases} 
X_t = a X_{t-1} + b Y_{t-6} + \varepsilon_t^x \\ 
Y_t = c Y_{t-1} + \varepsilon_t^y, 
\end{cases}
$$


Para evaluar la precisión y consistencia del estimador de Transfer Entropy en condiciones estocásticas controladas, se implementa un experimento estructurado en $20$ realizaciones independientes sobre series temporales de longitud $T = 10000$ del proceso autorregresivo bivariado. Sobre estas trayectorias, se calcula numéricamente la Transfer Entropy teórica bajo el supuesto gaussiano y se contrasta frente a las estimaciones empíricas obtenidas mediante el algoritmo KSG tanto en la dirección causal directa ($Y \to X$) como en la inversa ($X \to Y$). Este diseño permite cuantificar el error relativo medio de la estimación frente al valor analítico de referencia.

In [29]:
a, b, c = 0.5, 0.3, 0.7
sig_x2, sig_y2 = 1.0, 1.0
tau = 6
T = 10000
num_realizaciones = 20

lista_TE_teorica = []
lista_te_ksg = []
lista_te_ksg_inversa = []
lista_errores_relativos = []

for i in range(num_realizaciones):
    ex = np.random.normal(0, np.sqrt(sig_x2), T)
    ey = np.random.normal(0, np.sqrt(sig_y2), T)
    Z = np.zeros((T, 2))  # Z[:, 0] es X, Z[:, 1] es Y
    for t in range(tau, T):
        Z[t, 1] = c * Z[t-1, 1] + ey[t]
        Z[t, 0] = a * Z[t-1, 0] + b * Z[t-tau, 1] + ex[t]
    data = pd.DataFrame(Z, columns=['X', 'Y'])
    X = data['X'].values
    Y = data['Y'].values

    _, TE_teorica = GC_TE_autorreg_tau_gauss(a=a, b=b, c=c, sig_x2=sig_x2, sig_y2=sig_y2, tau=tau, datos=False)

    te_ksg_val = TE_JIDT(source=Y, target=X, n_neighbors=4, tau_source=tau)
    te_ksg_val_inversa = TE_JIDT(source=X, target=Y, n_neighbors=4, tau_source=tau)

    error_relativo_iter = abs(te_ksg_val - TE_teorica) / TE_teorica

    lista_TE_teorica.append(TE_teorica)
    lista_te_ksg.append(te_ksg_val)
    lista_te_ksg_inversa.append(te_ksg_val_inversa)
    lista_errores_relativos.append(error_relativo_iter)

media_TE_teorica = np.mean(lista_TE_teorica)
media_te_ksg = np.mean(lista_te_ksg)
media_te_ksg_inversa = np.mean(lista_te_ksg_inversa)
error_relativo_medio = np.mean(lista_errores_relativos)

print(f"--- Resultados medios tras {num_realizaciones} realizaciones (KSG) ---")
print(f"Transfer Entropy Teórica media (Gaussiana): {media_TE_teorica:.6f}")
print(f"Transfer Entropy Estimada media (KSG):      {media_te_ksg:.6f}")
print(f"Transfer Entropy Estimada inversa media (KSG): {media_te_ksg_inversa:.6f}")
print(f"Error relativo medio (promedio de los errores): {error_relativo_medio:.4%}")

--- Resultados medios tras 20 realizaciones (KSG) ---
Transfer Entropy Teórica media (Gaussiana): 0.072763
Transfer Entropy Estimada media (KSG):      0.071062
Transfer Entropy Estimada inversa media (KSG): 0.000258
Error relativo medio (promedio de los errores): 6.5926%
